In [ ]:
from pathlib import Path
import os
import json,numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor,RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.model_selection import KFold,train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostRegressor,Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'model_comparison'
OUT.mkdir(parents=True, exist_ok=True)
SPLIT,CV,SEED=10,42,42
TASK_TYPE = os.getenv('CATBOOST_TASK_TYPE', 'GPU').upper()
CATBOOST_DEVICE_ARGS = {'devices': '0'} if TASK_TYPE == 'GPU' else {}
F=['AM','AMc','Pr','Prc','Sp','Sp2','MSP','PM','CT','Ct','RT','Rt','RH','T','P','GHSV','Inert','H2/CO2']
C=['AM','Pr','Sp','Sp2','PM']; 
N=[x for x in F if x not in C]
d=pd.read_excel(ROOT/'database.xlsx')
X=d[F].copy()
y=pd.to_numeric(d['Rate'],errors='coerce')
ok=y.notna()
X,y=X.loc[ok].reset_index(drop=True),y.loc[ok].reset_index(drop=True)
X[C]=X[C].astype(str); xt,xs,yt,ys=train_test_split(X,y,test_size=.2,random_state=SPLIT)
folds=list(KFold(5,shuffle=True,random_state=CV).split(xt,yt))
print({'n':len(X),'train':len(xt),'test':len(xs),'split':SPLIT,'cv_seed':CV,'model_seed':SEED})

In [ ]:
def enc():
    try:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

pre = ColumnTransformer(
    transformers=[
        ('n', SimpleImputer(strategy='mean'), N),
        ('c', Pipeline([
            ('i', SimpleImputer(strategy='most_frequent')),
            ('o', enc())
        ]), C)
    ],
    verbose_feature_names_out=False
)

def p(m):
    return Pipeline([
        ('pre', clone(pre)),
        ('m', m)
    ])

models = {
    'RF': p(RandomForestRegressor(
        n_estimators=1000, 
        max_depth=24, 
        min_samples_split=2, 
        random_state=SEED, 
        n_jobs=-1
    )),
    'ET': p(ExtraTreesRegressor(
        n_estimators=500, 
        max_depth=24, 
        min_samples_split=2, 
        random_state=SEED, 
        n_jobs=-1
    )),
    'LGBM': p(LGBMRegressor(
        n_estimators=1500, 
        max_depth=12, 
        learning_rate=0.15, 
        random_state=SEED, 
        verbosity=-1, 
        n_jobs=-1
    )),
    'XGBoost': p(XGBRegressor(
        n_estimators=900, 
        max_depth=6, 
        learning_rate=0.05, 
        random_state=SEED, 
        objective='reg:squarederror', 
        n_jobs=-1
    )),
    'CatBoost-OHE': p(CatBoostRegressor(
        iterations=2000, 
        depth=10, 
        learning_rate=0.25, 
        random_seed=SEED, 
        loss_function='RMSE', 
        verbose=False, 
        allow_writing_files=False, 
        task_type=TASK_TYPE,
        **CATBOOST_DEVICE_ARGS
    ))
}

nce = {
    'iterations': 2000,
    'learning_rate': 0.4,
    'depth': 10,
    'l2_leaf_reg': 9.88,
    'bagging_temperature': 0.89,
    'border_count': 128,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': SEED,
    'verbose': False,
    'allow_writing_files': False,
    'task_type': TASK_TYPE
}

def sc(a, b):
    return mean_squared_error(a, b, squared=False), r2_score(a, b)
if TASK_TYPE == 'GPU':
    nce.update(CATBOOST_DEVICE_ARGS)


In [ ]:
rows = []
final = []

for name, e in models.items():
    for k, (a, b) in enumerate(folds, 1):
        m = clone(e).fit(xt.iloc[a], yt.iloc[a])
        q, z = sc(yt.iloc[b], m.predict(xt.iloc[b]))
        rows.append({
            'Model': name, 
            'Fold': k, 
            'Train_n': len(a), 
            'Validation_n': len(b), 
            'Validation_RMSE': q, 
            'Validation_R2': z
        })
        
    m = clone(e).fit(xt, yt)
    q, z = sc(yt, m.predict(xt))
    v, w = sc(ys, m.predict(xs))
    final.append({
        'Model': name, 
        'Train_RMSE': q, 
        'Train_R2': z, 
        'Test_RMSE': v, 
        'Test_R2': w
    })

for k, (a, b) in enumerate(folds, 1):
    m = CatBoostRegressor(**nce)
    A = Pool(xt.iloc[a], yt.iloc[a], cat_features=C)
    B = Pool(xt.iloc[b], yt.iloc[b], cat_features=C)
    m.fit(A)
    q, z = sc(yt.iloc[b], m.predict(B))
    rows.append({
        'Model': 'CatBoost-NCE-model424', 
        'Fold': k, 
        'Train_n': len(a), 
        'Validation_n': len(b), 
        'Validation_RMSE': q, 
        'Validation_R2': z
    })

m = CatBoostRegressor(**nce)
A = Pool(xt, yt, cat_features=C)
B = Pool(xs, ys, cat_features=C)
m.fit(A)
q, z = sc(yt, m.predict(A))
v, w = sc(ys, m.predict(B))
final.append({
    'Model': 'CatBoost-NCE-model424', 
    'Train_RMSE': q, 
    'Train_R2': z, 
    'Test_RMSE': v, 
    'Test_R2': w
})

In [ ]:
fm = pd.DataFrame(rows).sort_values(['Model', 'Fold'])

sm = fm.groupby('Model', as_index=False).agg(
    Mean_CV_RMSE=('Validation_RMSE', 'mean'),
    SD_CV_RMSE=('Validation_RMSE', 'std'),
    Mean_CV_R2=('Validation_R2', 'mean'),
    SD_CV_R2=('Validation_R2', 'std')
).merge(
    pd.DataFrame(final), on='Model'
).sort_values('Mean_CV_RMSE').reset_index(drop=True)

sm.insert(0, 'CV_Rank_by_RMSE', np.arange(1, 7))
sm['Test_Rank_by_R2'] = sm.Test_R2.rank(ascending=False, method='min').astype(int)

fm.to_csv(
    OUT / 'six_model_split2026_shuffled_fold_metrics.csv', 
    index=False, 
    encoding='utf-8-sig'
)

sm.to_csv(
    OUT / 'six_model_split2026_shuffled_summary.csv', 
    index=False, 
    encoding='utf-8-sig'
)

manifest = {
    'data': str(ROOT / 'database.xlsx'),
    'split_seed': SPLIT,
    'cv': {
        'n_splits': 5,
        'shuffle': True,
        'random_state': CV
    },
    'model_seed': SEED,
    'nce_parameters': nce
}

with open(OUT / 'run_manifest.json', 'w', encoding='utf8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

display(sm.round(4))